In [1]:
import sys
from pathlib import Path

import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd().parents[1]
sys.path.append(str(project_root))

from src.visualization import check_float_binary_columns

# Diabetes prediction - Preprocessing (BRFSS)

## Load data

In [2]:
brfss_data = pd.read_csv("../../data/processed/BRFSS2015_binary_data_cleaned.csv")

In [3]:
brfss_data

,diabetes,high_blood_pressure,high_cholesterol,chol_checked_recently,bmi,smoked_at_least_100_cigarettes,had_stroke,heart_disease_or_attack,physically_active,consumes_fruit_daily,...,has_healthcare_coverage,skipped_doctor_due_to_cost,general_health,mental_health_bad_days,physical_health_bad_days,difficulty_walking,sex,age,education_level,income_level
0,0,1.0,1.0,1.0,40.18,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,63.0,4.0,3.0
1,0,0.0,0.0,0.0,25.09,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,52.0,6.0,1.0
2,0,1.0,1.0,1.0,28.19,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,63.0,4.0,8.0
3,0,1.0,0.0,1.0,26.52,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,73.0,3.0,6.0
4,0,1.0,1.0,1.0,23.89,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,70.0,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
252485,0,1.0,1.0,1.0,45.00,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,5.0,0.0,1.0,42.0,6.0,7.0
252486,1,1.0,1.0,1.0,18.42,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,72.0,2.0,4.0
252487,0,0.0,0.0,1.0,28.34,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,29.0,5.0,2.0
252488,0,1.0,0.0,1.0,23.15,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,52.0,5.0,1.0


## Train/Test Split

We start with the splitting of the data into training and testing split. This is done before the scaling in order to avoid data leakage.

In [4]:
X = brfss_data.drop(columns=["diabetes"])
y = brfss_data["diabetes"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [6]:
print(f"Train shape: {X_train.shape}, positive rate: {y_train.mean():.4f}")
print(f"Test shape: {X_test.shape}, positive rate: {y_test.mean():.4f}")

Train shape: (201992, 21), positive rate: 0.1583
Test shape: (50498, 21), positive rate: 0.1583


## Feature Selection

The EDA notebook flagged `has_healthcare_coverage`, `chol_checked_recently`, and `had_stroke` as near-constant candidates based on a >95%/<5% prevalence threshold. However, prevalence alone is not sufficient justification for removal, since a rare or near-universal category can still carry strong predictive signal. Each candidate is checked against the target below before a final decision is made.

For each candidate, the diabetes rate is compared between groups. A large difference indicates the feature is discriminative despite its low variance in raw prevalence. A small difference confirms it is safe to remove.

In [7]:
print(pd.crosstab(brfss_data["had_stroke"], brfss_data["diabetes"], normalize="index") * 100)
print(pd.crosstab(brfss_data["has_healthcare_coverage"], brfss_data["diabetes"], normalize="index") * 100)
print(pd.crosstab(brfss_data["chol_checked_recently"], brfss_data["diabetes"], normalize="index") * 100)

diabetes            0          1
had_stroke                      
0.0         84.954872  15.045128
1.0         65.672367  34.327633
diabetes                         0          1
has_healthcare_coverage                      
0.0                      86.502376  13.497624
1.0                      84.048185  15.951815
diabetes                       0          1
chol_checked_recently                      
0.0                    96.797379   3.202621
1.0                    83.677257  16.322743


`had_stroke` shows a substantial difference in diabetes rate between groups (15.05% vs 34.33%, ~19.3 percentage points), consistent with stroke being an established diabetes comorbidity. Despite its low prevalence (4.08%), it is retained.

`has_healthcare_coverage` shows only a small difference (13.50% vs. 15.95%, ~2.45 percentage points) and is removed.

`chol_checked_recently` shows a large difference (3.20% vs 16.32%, ~13.1 percentage points) despite its high prevalence, and is therefore retained rather than removed. This large difference likely reflects a detection/ascertainment effect - individuals who never have their cholesterol checked are also less likely to have been screened and diagnosed for diabetes - rather than the feature being a genuine clinical risk factor. It should not be interpreted causally, even though it carries real predictive signal.

Only `has_healthcare_coverage` is removed as a result of this check.

In [8]:
low_variance_cols = ["has_healthcare_coverage"]

X_train = X_train.drop(columns=low_variance_cols)
X_test = X_test.drop(columns=low_variance_cols)

## Encoding

Unlike the other dataset, BRFSS contains no string categorical columns requiring one-hot encoding - all features are already numeric (binary flags or ordinal/continuous values). This section instead identifies which columns are binary versus continuous, to determine which require scaling in the next step.

In [9]:
binary_cols, float_cols = check_float_binary_columns(X_train)

In [10]:
categorical_attributes = X_train.drop(columns=float_cols)

In [11]:
is_perfectly_binary = all(
    set(categorical_attributes[col].dropna().unique()).issubset({0.0, 1.0}) 
    for col in categorical_attributes.columns
)

In [12]:
print(is_perfectly_binary)

True


There are no categorical attribites. Therefore, we don't need encoding.

## Scaling

In [13]:
X_train[float_cols].describe()

,bmi,general_health,mental_health_bad_days,physical_health_bad_days,age,education_level,income_level
count,201992.000000,201992.000000,201992.000000,201992.00000,201992.000000,201992.000000,201992.000000
mean,28.410469,2.514847,3.186354,4.26005,57.018892,5.047928,6.049155
std,6.607033,1.066338,7.412606,8.73395,15.104939,0.985393,2.069344
min,12.050000,1.000000,0.000000,0.00000,18.000000,1.000000,1.000000
25%,24.140000,2.000000,0.000000,0.00000,47.000000,4.000000,5.000000
50%,27.370000,2.000000,0.000000,0.00000,59.000000,5.000000,7.000000
75%,31.320000,3.000000,2.000000,3.00000,68.000000,6.000000,8.000000
max,97.650000,5.000000,30.000000,30.00000,80.000000,6.000000,8.000000


In [14]:
binary_cols, continuous_cols = check_float_binary_columns(X_train, binary=True, continuous=True)

Unlike the other dataset, no feature here showed the kind of extreme, artificial outlier concentration that justified RobustScaler. The EDA notebook found `bmi` here to be realistically right-skewed without evidence of placeholder imputation. `StandardScaler` is therefore applied uniformly to all continuous columns.

In [15]:
preprocessor = make_column_transformer(
    (StandardScaler(), continuous_cols),
    remainder="passthrough",
    verbose_feature_names_out=False
)

In [16]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [17]:
feature_names = preprocessor.get_feature_names_out()
X_train_transformed = pd.DataFrame(X_train_transformed, columns=feature_names, index=X_train.index)
X_test_transformed = pd.DataFrame(X_test_transformed, columns=feature_names, index=X_test.index)

In [18]:
X_train_transformed

,bmi,general_health,mental_health_bad_days,physical_health_bad_days,age,education_level,income_level,high_blood_pressure,high_cholesterol,chol_checked_recently,smoked_at_least_100_cigarettes,had_stroke,heart_disease_or_attack,physically_active,consumes_fruit_daily,consumes_veggies_daily,heavy_alcohol_consumption,skipped_doctor_due_to_cost,difficulty_walking,sex
227852,-1.347124,-0.482819,-0.429857,-0.487759,0.594582,-1.063465,-0.507000,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
44290,-0.900629,0.454972,-0.429857,-0.487759,-2.318379,0.966188,0.942738,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0
164725,0.505452,2.330554,-0.429857,2.947122,1.124212,-2.078291,-0.990246,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
213253,-0.390565,-0.482819,-0.429857,-0.487759,0.660786,0.966188,0.942738,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
33474,1.284926,1.392763,3.617312,1.229681,-0.597084,-0.048638,0.459492,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111661,-0.319428,-0.482819,-0.429857,-0.487759,0.991804,-0.048638,0.459492,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
200025,0.794539,0.454972,-0.429857,-0.487759,0.395971,-1.063465,-0.507000,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
167549,-0.033369,0.454972,-0.429857,-0.487759,1.124212,0.966188,-1.956738,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0
211029,-0.295212,-1.420610,-0.429857,0.313713,-0.067454,0.966188,0.942738,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0


In [19]:
X_test_transformed

,bmi,general_health,mental_health_bad_days,physical_health_bad_days,age,education_level,income_level,high_blood_pressure,high_cholesterol,chol_checked_recently,smoked_at_least_100_cigarettes,had_stroke,heart_disease_or_attack,physically_active,consumes_fruit_daily,consumes_veggies_daily,heavy_alcohol_consumption,skipped_doctor_due_to_cost,difficulty_walking,sex
123718,0.096796,2.330554,3.617312,2.832626,0.726990,-1.063465,-1.956738,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
104825,-0.949062,-0.482819,-0.429857,-0.487759,0.991804,-1.063465,0.459492,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
95947,-0.546460,-1.420610,-0.429857,-0.487759,-1.590139,-0.048638,0.942738,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
241048,-0.143858,-0.482819,0.244671,-0.487759,-1.656343,0.966188,-0.023754,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
215239,-0.413268,0.454972,-0.160046,-0.487759,0.925601,0.966188,0.942738,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155467,0.169446,-0.482819,-0.429857,-0.487759,-0.133658,0.966188,0.942738,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
248350,-1.312312,0.454972,-0.429857,-0.487759,1.058008,-0.048638,-0.507000,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
54350,0.420694,-1.420610,-0.429857,-0.487759,-1.656343,0.966188,0.942738,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0
250656,0.075606,1.392763,-0.429857,-0.487759,0.793193,-0.048638,-0.990246,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0


## Save Datasets

In [20]:
X_train_transformed.to_csv("../../data/processed/brfss_data_train.csv", index=False)

In [21]:
X_test_transformed.to_csv("../../data/processed/brfss_data_test.csv", index=False)

In [22]:
y_train.to_csv("../../data/processed/brfss_data_y_train.csv", index=False)

In [23]:
y_test.to_csv("../../data/processed/brfss_data_y_test.csv", index=False)